# 🔥 CRIAS — Gerador de 3D

### Só faça isto:

**1.** No menu de cima: `Ambiente de execução` → `Alterar o tipo de ambiente` → escolha **T4 GPU** → `Salvar`

**2.** No menu de cima: `Ambiente de execução` → `Executar tudo`

**3.** Espere ~4 minutos. Vai aparecer um **botão para escolher as imagens** — escolha (pode várias).

**4.** Os arquivos `.glb` baixam sozinhos no final.

> Dica: nomeie a imagem com o nome que quer no jogo (ex: `aerix.png` → vira `aerix.glb`)

In [ ]:
#@title ⚙️ Preparando (leva ~4 min — pode ignorar os textos que aparecem)
import os, sys, subprocess
print('Instalando... aguarde, isso é normal demorar.')
if not os.path.exists('/content/TripoSR'):
    subprocess.run('git clone --depth 1 -q https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR', shell=True)
subprocess.run(f'{sys.executable} -m pip install -q -r /content/TripoSR/requirements.txt onnxruntime rembg', shell=True)
os.chdir('/content/TripoSR')
try:
    import torch
    gpu = torch.cuda.is_available()
except Exception:
    gpu = False
print('\n✅ PRONTO!' if gpu else '\n⚠️ SEM GPU: vá em Ambiente de execução → Alterar o tipo → T4 GPU, e rode tudo de novo.')
print('Agora role para baixo até aparecer o botão de escolher arquivos.')

In [ ]:
#@title 📤 Escolha as imagens (clique em 'Escolher arquivos' abaixo)
from google.colab import files
from PIL import Image
import os, glob, shutil

os.makedirs('/content/in', exist_ok=True)
for f in glob.glob('/content/in/*'):
    os.remove(f)

print('👇 Clique no botão e escolha as imagens dos monstros/personagens')
up = files.upload()

imgs = []
for name, data in up.items():
    p = '/content/in/' + name
    open(p, 'wb').write(data)
    # tira o fundo verde chroma automaticamente
    im = Image.open(p).convert('RGBA')
    px = im.load()
    for y in range(im.height):
        for x in range(im.width):
            r, g, b, a = px[x, y]
            if g > 90 and g > r * 1.35 and g > b * 1.35:
                px[x, y] = (0, 0, 0, 0)
            elif g > max(r, b):
                px[x, y] = (r, max(r, b), b, a)
    im.save(p)
    imgs.append(p)
print(f'\n✅ {len(imgs)} imagem(ns) recebida(s). Continue rolando para baixo.')

In [ ]:
#@title 🎲 Gerando os modelos 3D e baixando (~1 min por imagem)
import os, sys, glob, shutil, subprocess
from google.colab import files as gfiles

os.chdir('/content/TripoSR')
os.makedirs('/content/glb', exist_ok=True)
prontos = []

for p in sorted(glob.glob('/content/in/*')):
    nome = os.path.splitext(os.path.basename(p))[0].lower().replace(' ', '-')
    print(f'▶ gerando {nome} ...')
    subprocess.run(
        f'{sys.executable} run.py "{p}" --output-dir "/content/out/{nome}" '
        f'--model-save-format glb --bake-texture --texture-resolution 1024',
        shell=True)
    achados = glob.glob(f'/content/out/{nome}/**/*.glb', recursive=True)
    if achados:
        destino = f'/content/glb/{nome}.glb'
        shutil.copy(achados[0], destino)
        prontos.append(destino)
        print(f'   ✅ {nome}.glb pronto')
    else:
        print(f'   ❌ falhou em {nome}')

print(f'\n{"="*40}\n✅ {len(prontos)} modelo(s) pronto(s)! Baixando...\n{"="*40}')
for g in prontos:
    gfiles.download(g)
print('\nPronto! Agora é só mandar os arquivos .glb para o Claude integrar no jogo.')